# Football Match Outcome Prediction

Predicts **home win / draw / away win** from pre-match information only,
on the Transfermarkt domestic-league core (`../clean/`, seasons 2012-2025 —
see `../clean/README.md` for how that core was audited and built).

Single self-contained notebook: raw-data quality checks -> leakage-safe
feature engineering -> sklearn `Pipeline`s -> time-series cross-validated
hyperparameter search across four algorithms -> evaluation -> predictions.

**Environment note:** this machine's Windows Smart App Control policy blocks
scikit-learn's compiled DLLs under the default Python install; a second
Python 3.14 install has wheels that pass the check, so this notebook runs on
that kernel (`py314`, registered via `python -m ipykernel install --user
--name py314`). Elsewhere, any normal Python with `requirements.txt`
installed runs this unmodified.

In [1]:
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, log_loss, confusion_matrix,
                              classification_report, f1_score)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)
pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 50)

RAW = "../clean"
LABEL = {"H": "Home win", "D": "Draw", "A": "Away win"}
CLASSES = ["A", "D", "H"]
CLASS_IDX = {c: i for i, c in enumerate(CLASSES)}
RNG = 42

## 1. Load raw data & quality checks

Working from `../clean/` — already audited for FK integrity and PK
uniqueness (see `../clean/README.md`) — but re-verifying the invariants
this pipeline actually depends on is standard practice, not redundant:
a prior audit checked *that* dataset for *its* purposes, this checks the
exact columns this model consumes.

In [2]:
games = pd.read_csv(f"{RAW}/games.csv", parse_dates=["date"])


**Leakage check — why `home_club_position`/`away_club_position` are never used below.**
If these were pre-match standings, every club would be tied on matchday 1 of a season.
They aren't — proof they're actually post-match standings, and using them would leak
the very outcome being predicted.

In [3]:
games.shape


(63382, 23)

In [4]:
games.isnull().sum()

game_id                      0
competition_id               0
season                       0
round                        0
date                         0
home_club_id                 0
away_club_id                 0
home_club_goals              0
away_club_goals              0
home_club_position          73
away_club_position          73
home_club_manager_name     173
away_club_manager_name     173
stadium                     41
attendance                6211
referee                    117
url                          0
home_club_formation       5022
away_club_formation       5021
home_club_name               0
away_club_name               0
aggregate                    0
competition_type             0
dtype: int64

In [5]:
df = games.drop(["home_club_manager_name", "away_club_manager_name"], axis=1)


In [6]:
len(df['game_id'])

63382

In [7]:
df.isnull().sum()

game_id                   0
competition_id            0
season                    0
round                     0
date                      0
home_club_id              0
away_club_id              0
home_club_goals           0
away_club_goals           0
home_club_position       73
away_club_position       73
stadium                  41
attendance             6211
referee                 117
url                       0
home_club_formation    5022
away_club_formation    5021
home_club_name            0
away_club_name            0
aggregate                 0
competition_type          0
dtype: int64

In [8]:
df = games.drop(["attendance", "home_club_formation",'away_club_formation'], axis=1)


In [9]:
df.isnull().sum()

game_id                     0
competition_id              0
season                      0
round                       0
date                        0
home_club_id                0
away_club_id                0
home_club_goals             0
away_club_goals             0
home_club_position         73
away_club_position         73
home_club_manager_name    173
away_club_manager_name    173
stadium                    41
referee                   117
url                         0
home_club_name              0
away_club_name              0
aggregate                   0
competition_type            0
dtype: int64

In [10]:
df[df['home_club_position'].isnull()]

,game_id,competition_id,season,round,date,home_club_id,away_club_id,home_club_goals,away_club_goals,home_club_position,away_club_position,home_club_manager_name,away_club_manager_name,stadium,referee,url,home_club_name,away_club_name,aggregate,competition_type
11800,2518620,GR1,2014,26. Matchday,2015-03-18,2672,28956,3,0,NaN,NaN,Apostolos Mantzios,Vangelis Vlachos,Stadio Livadias,Konstantinos Ioannidis,https://www.transfermarkt.co.uk/apo-levadiakos...,APO Levadiakos Football Club,AEL Kalloni,3:0,domestic_league
11801,2518621,GR1,2014,26. Matchday,2015-03-18,653,7185,0,1,NaN,NaN,NaN,NaN,Gipedo Theodoros Vardinogiannis,Christos Mitsios,https://www.transfermarkt.co.uk/ofi-crete-fc_p...,Omilos Filathlon Irakliou FC,Panthrakikos Komotini,0:1,domestic_league
11802,2518622,GR1,2014,26. Matchday,2015-03-18,1091,5220,1,0,NaN,NaN,Georgios Georgiadis,Giannis Taousianis,Toumba Stadium,Georgios Kominis,https://www.transfermarkt.co.uk/paok-thessalon...,Panthessalonikios Athlitikos Omilos Konstantin...,GS Ergotelis,1:0,domestic_league
11803,2518623,GR1,2014,26. Matchday,2015-03-18,128,6418,3,0,NaN,NaN,Răzvan Lucescu,Makis Chavos,Xanthi Arena,Michalis Koukoulakis,https://www.transfermarkt.co.uk/skoda-xanthi_p...,AO Xanthi,Panetolikos Agrinio,3:0,domestic_league
11804,2518624,GR1,2014,26. Matchday,2015-03-17,5219,5572,3,0,NaN,NaN,NaN,NaN,Ethniko Stadio Kerkyras,NaN,https://www.transfermarkt.co.uk/aok-kerkyra_ni...,AOK Kerkyra,Niki Volou,3:0,domestic_league
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62462,4798378,ARG1,2025,9. Matchday,2026-05-03,209,14554,0,1,NaN,NaN,Eduardo Coudet,Julio César Falcioni,Mâs Monumental,Yael Falcón Pérez,https://www.transfermarkt.co.uk/ca-river-plate...,Club Atlético River Plate,Club Atlético Tucumán,0:1,domestic_league
62463,4798379,ARG1,2025,9. Matchday,2026-05-03,12301,12179,1,1,NaN,NaN,Israel Damonte,Alfredo Berti,José María Minella,Pablo Echavarría,https://www.transfermarkt.co.uk/ca-aldosivi_cs...,Club Atlético Aldosivi,Club Sportivo Independiente Rivadavia,1:1,domestic_league
62464,4798380,ARG1,2025,9. Matchday,2026-05-02,25184,830,1,2,NaN,NaN,Rubén Insúa,Pedro Troglio,Claudio Chiqui Tapia,Darío Herrera,https://www.transfermarkt.co.uk/ca-barracas-ce...,Club Atlético Barracas Central,Club Atlético Banfield,1:2,domestic_league
62465,4798381,ARG1,2025,9. Matchday,2026-05-03,1106,1030,2,0,NaN,NaN,Ariel Pereyra,Nicolás Diez,Juan Carmelo Zerillo,Juan Pafundi,https://www.transfermarkt.co.uk/club-de-gimnas...,Club de Gimnasia y Esgrima La Plata,Asociación Atlética Argentinos Juniors,2:0,domestic_league


In [11]:
df['competition_id'].value_counts()

competition_id
GB1     5320
ES1     5320
IT1     5320
FR1     4997
TR1     4618
L1      4284
NL1     4210
PO1     4152
BE1     3550
RU1     3360
GR1     3085
SC1     2903
UKR1    2700
DK1     2312
MLS1     727
PL1      612
SA1      612
BRA1     557
SER1     480
TS1      480
RO1      480
C1       430
ARG1     418
A1       356
KR1      348
NO1      329
SE1      329
AUS1     325
JAP1     296
MEX1     241
RSK1     231
Name: count, dtype: int64

In [12]:
compi_1= df[df['competition_id']=='GB1']

In [13]:
compi_1.isnull().sum()

game_id                   0
competition_id            0
season                    0
round                     0
date                      0
home_club_id              0
away_club_id              0
home_club_goals           0
away_club_goals           0
home_club_position        0
away_club_position        0
home_club_manager_name    0
away_club_manager_name    0
stadium                   0
referee                   0
url                       0
home_club_name            0
away_club_name            0
aggregate                 0
competition_type          0
dtype: int64

In [14]:
compi_1

,game_id,competition_id,season,round,date,home_club_id,away_club_id,home_club_goals,away_club_goals,home_club_position,away_club_position,home_club_manager_name,away_club_manager_name,stadium,referee,url,home_club_name,away_club_name,aggregate,competition_type
1256,2225443,GB1,2012,2. Matchday,2012-12-11,289,1032,3,0,5.0,16.0,Martin O'Neill,Brian McDermott,Stadium of Light,Neil Swarbrick,https://www.transfermarkt.co.uk/sunderland-afc...,Sunderland AFC,Reading FC,3:0,domestic_league
1257,2225444,GB1,2012,2. Matchday,2012-08-25,2288,379,3,0,1.0,11.0,Michael Laudrup,Sam Allardyce,Swansea.com Stadium,Martin Atkinson,https://www.transfermarkt.co.uk/swansea-city_w...,Swansea City,West Ham United,3:0,domestic_league
1258,2225445,GB1,2012,2. Matchday,2012-08-25,405,29,1,3,20.0,3.0,Paul Lambert,David Moyes,Villa Park,Michael Oliver,https://www.transfermarkt.co.uk/aston-villa_ev...,Aston Villa,Everton FC,1:3,domestic_league
1259,2225446,GB1,2012,2. Matchday,2012-08-25,180,1071,0,2,19.0,9.0,Nigel Adkins,Roberto Martínez,St Mary's Stadium,Anthony Taylor,https://www.transfermarkt.co.uk/southampton-fc...,Southampton FC,Wigan Athletic,0:2,domestic_league
1260,2225447,GB1,2012,2. Matchday,2012-08-25,985,931,3,2,8.0,7.0,Sir Alex Ferguson,Martin Jol,Old Trafford,Kevin Friend,https://www.transfermarkt.co.uk/manchester-uni...,Manchester United,Fulham FC,3:2,domestic_league
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57731,4626173,GB1,2025,38. Matchday,2026-05-24,281,405,1,2,2.0,4.0,Pep Guardiola,Unai Emery,Etihad Stadium,Andrew Madley,https://www.transfermarkt.co.uk/manchester-cit...,Manchester City,Aston Villa,1:2,domestic_league
57732,4626174,GB1,2025,38. Matchday,2026-05-24,703,989,1,1,16.0,6.0,Vítor Pereira,Andoni Iraola,The City Ground,Craig Pawson,https://www.transfermarkt.co.uk/nottingham-for...,Nottingham Forest,AFC Bournemouth,1:1,domestic_league
57733,4626175,GB1,2025,38. Matchday,2026-05-24,289,631,2,1,7.0,10.0,Régis Le Bris,Calum McFarlane,Stadium of Light,Chris Kavanagh,https://www.transfermarkt.co.uk/sunderland-afc...,Sunderland AFC,Chelsea FC,2:1,domestic_league
57734,4626176,GB1,2025,38. Matchday,2026-05-24,148,29,1,0,17.0,13.0,Roberto De Zerbi,David Moyes,Tottenham Hotspur Stadium,Michael Oliver,https://www.transfermarkt.co.uk/tottenham-hots...,Tottenham Hotspur,Everton FC,1:0,domestic_league


In [15]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Feature Prep
compi_1 = compi_1.copy()
compi_1["outcome"] = np.where(compi_1["home_club_goals"] > compi_1["away_club_goals"], "Home Win",
                     np.where(compi_1["home_club_goals"] < compi_1["away_club_goals"], "Away Win", "Draw"))
compi_1["total_goals"] = compi_1["home_club_goals"] + compi_1["away_club_goals"]
compi_1["scoreline"] = compi_1["home_club_goals"].astype(str) + " - " + compi_1["away_club_goals"].astype(str)

# 2. Season-by-Season Outcome Trend
season_trends = compi_1.groupby(["season", "outcome"]).size().unstack(fill_value=0)
season_pcts = (season_trends.div(season_trends.sum(axis=1), axis=0) * 100).reset_index()

fig1 = px.line(
    season_pcts, x="season", y=["Home Win", "Away Win", "Draw"],
    markers=True, title="Premier League: Outcome Rates by Season (Notice COVID 2020 Inversion)",
    labels={"value": "Percentage (%)", "season": "Season", "variable": "Outcome"},
    color_discrete_map={"Home Win": "#1f77b4", "Away Win": "#d62728", "Draw": "#7f7f7f"}
)
fig1.add_vrect(x0=2019.5, x1=2020.5, annotation_text="COVID Behind-Closed-Doors", fillcolor="red", opacity=0.1)
fig1.show()

# 3. Top 10 Common Scorelines Bar Chart


In [16]:
top_scores = compi_1["scoreline"].value_counts().head(10).reset_index()
top_scores.columns = ["Scoreline", "Frequency"]

fig2 = px.bar(
    top_scores, x="Scoreline", y="Frequency", color="Frequency",
    color_continuous_scale="Blues", title="Top 10 Most Common Premier League Scorelines (2012–2025)",
    text="Frequency"
)
fig2.show()



In [17]:
# 4. Cumulative All-Time Points (Big Six vs Rest)
h_pts = compi_1.groupby("home_club_name").apply(lambda d: ((d["outcome"]=="Home Win")*3 + (d["outcome"]=="Draw")*1).sum())
a_pts = compi_1.groupby("away_club_name").apply(lambda d: ((d["outcome"]=="Away Win")*3 + (d["outcome"]=="Draw")*1).sum())
total_pts = (h_pts + a_pts).sort_values(ascending=False).head(10).reset_index()
total_pts.columns = ["Club", "Points"]

fig3 = px.bar(
    total_pts, x="Points", y="Club", orientation="h",
    color="Points", color_continuous_scale="Viridis",
    title="Top 10 Clubs: Cumulative Premier League Points (2012–2025)"
)
fig3.update_layout(yaxis={'categoryorder':'total ascending'})
fig3.show()


In [18]:
compi_1

,game_id,competition_id,season,round,date,home_club_id,away_club_id,home_club_goals,away_club_goals,home_club_position,away_club_position,home_club_manager_name,away_club_manager_name,stadium,referee,url,home_club_name,away_club_name,aggregate,competition_type,outcome,total_goals,scoreline
1256,2225443,GB1,2012,2. Matchday,2012-12-11,289,1032,3,0,5.0,16.0,Martin O'Neill,Brian McDermott,Stadium of Light,Neil Swarbrick,https://www.transfermarkt.co.uk/sunderland-afc...,Sunderland AFC,Reading FC,3:0,domestic_league,Home Win,3,3 - 0
1257,2225444,GB1,2012,2. Matchday,2012-08-25,2288,379,3,0,1.0,11.0,Michael Laudrup,Sam Allardyce,Swansea.com Stadium,Martin Atkinson,https://www.transfermarkt.co.uk/swansea-city_w...,Swansea City,West Ham United,3:0,domestic_league,Home Win,3,3 - 0
1258,2225445,GB1,2012,2. Matchday,2012-08-25,405,29,1,3,20.0,3.0,Paul Lambert,David Moyes,Villa Park,Michael Oliver,https://www.transfermarkt.co.uk/aston-villa_ev...,Aston Villa,Everton FC,1:3,domestic_league,Away Win,4,1 - 3
1259,2225446,GB1,2012,2. Matchday,2012-08-25,180,1071,0,2,19.0,9.0,Nigel Adkins,Roberto Martínez,St Mary's Stadium,Anthony Taylor,https://www.transfermarkt.co.uk/southampton-fc...,Southampton FC,Wigan Athletic,0:2,domestic_league,Away Win,2,0 - 2
1260,2225447,GB1,2012,2. Matchday,2012-08-25,985,931,3,2,8.0,7.0,Sir Alex Ferguson,Martin Jol,Old Trafford,Kevin Friend,https://www.transfermarkt.co.uk/manchester-uni...,Manchester United,Fulham FC,3:2,domestic_league,Home Win,5,3 - 2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57731,4626173,GB1,2025,38. Matchday,2026-05-24,281,405,1,2,2.0,4.0,Pep Guardiola,Unai Emery,Etihad Stadium,Andrew Madley,https://www.transfermarkt.co.uk/manchester-cit...,Manchester City,Aston Villa,1:2,domestic_league,Away Win,3,1 - 2
57732,4626174,GB1,2025,38. Matchday,2026-05-24,703,989,1,1,16.0,6.0,Vítor Pereira,Andoni Iraola,The City Ground,Craig Pawson,https://www.transfermarkt.co.uk/nottingham-for...,Nottingham Forest,AFC Bournemouth,1:1,domestic_league,Draw,2,1 - 1
57733,4626175,GB1,2025,38. Matchday,2026-05-24,289,631,2,1,7.0,10.0,Régis Le Bris,Calum McFarlane,Stadium of Light,Chris Kavanagh,https://www.transfermarkt.co.uk/sunderland-afc...,Sunderland AFC,Chelsea FC,2:1,domestic_league,Home Win,3,2 - 1
57734,4626176,GB1,2025,38. Matchday,2026-05-24,148,29,1,0,17.0,13.0,Roberto De Zerbi,David Moyes,Tottenham Hotspur Stadium,Michael Oliver,https://www.transfermarkt.co.uk/tottenham-hots...,Tottenham Hotspur,Everton FC,1:0,domestic_league,Home Win,1,1 - 0


In [19]:
df['home_club_id'].value_counts(),df['home_club_name'].value_counts()

(home_club_id
 985     266
 148     266
 631     266
 31      266
 281     266
        ... 
 3654      6
 353       5
 267       5
 5202      5
 1118      5
 Name: count, Length: 773, dtype: int64,
 home_club_name
 Manchester United         266
 Tottenham Hotspur         266
 Chelsea FC                266
 Liverpool FC              266
 Manchester City           266
                          ... 
 Kalmar Fotbollförening      6
 Lillestrøm Sportsklubb      5
 Idrettsklubben Start        5
 Västerås Sportklubb FK      5
 Örgryte IS                  5
 Name: count, Length: 773, dtype: int64)

In [20]:
# List of your desired columns (in logical order)
selected_columns = [
    # Match Date & Context
    "season",
    "date",
    "stadium",
    "competition_type",
    
    # Teams & Managers
    "home_club_name",
    "away_club_name",
    "home_club_manager_name",
    "away_club_manager_name",
    
    # Outcomes & Scores
    "scoreline",
    "aggregate",
    "total_goals",
    "outcome"
]

# Separate into a new clean DataFrame
df_separated = compi_1[selected_columns].copy()

# Preview
df_separated.head()


,season,date,stadium,competition_type,home_club_name,away_club_name,home_club_manager_name,away_club_manager_name,scoreline,aggregate,total_goals,outcome
1256,2012,2012-12-11,Stadium of Light,domestic_league,Sunderland AFC,Reading FC,Martin O'Neill,Brian McDermott,3 - 0,3:0,3,Home Win
1257,2012,2012-08-25,Swansea.com Stadium,domestic_league,Swansea City,West Ham United,Michael Laudrup,Sam Allardyce,3 - 0,3:0,3,Home Win
1258,2012,2012-08-25,Villa Park,domestic_league,Aston Villa,Everton FC,Paul Lambert,David Moyes,1 - 3,1:3,4,Away Win
1259,2012,2012-08-25,St Mary's Stadium,domestic_league,Southampton FC,Wigan Athletic,Nigel Adkins,Roberto Martínez,0 - 2,0:2,2,Away Win
1260,2012,2012-08-25,Old Trafford,domestic_league,Manchester United,Fulham FC,Sir Alex Ferguson,Martin Jol,3 - 2,3:2,5,Home Win


In [21]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Target variable
compi_1 = compi_1.copy()
compi_1["outcome"] = np.where(
    compi_1["home_club_goals"] > compi_1["away_club_goals"],
    "Home Win",
    np.where(compi_1["home_club_goals"] < compi_1["away_club_goals"], "Away Win", "Draw"),
)

# 2. Define the exact features
cat_features = [
    "stadium",
    "competition_type",
    "home_club_name",
    "away_club_name",
    "home_club_manager_name",
    "away_club_manager_name",
]
num_features = ["season"]
feature_cols = cat_features + num_features

# Clean missing values
for col in cat_features:
    compi_1[col] = compi_1[col].fillna("Unknown").astype(str)

X = compi_1[feature_cols]
y = compi_1["outcome"]

# 3. Time-based train/test split (Train: up to 2023, Test: 2024-2025)
train_mask = compi_1["season"] < 2024
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]

# 4. Scikit-Learn Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
        ("num", StandardScaler(), num_features),
    ]
)

match_predictor = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, C=0.5, random_state=42)),
    ]
)

# Train the model
match_predictor.fit(X_train, y_train)

# Evaluate on held-out test seasons (2024-2025)
y_pred = match_predictor.predict(X_test)
print(f"Model Test Accuracy (Seasons 2024-2025): {accuracy_score(y_test, y_pred):.1%}\n")
print(classification_report(y_test, y_pred))


Model Test Accuracy (Seasons 2024-2025): 45.4%

              precision    recall  f1-score   support

    Away Win       0.43      0.40      0.41       246
        Draw       0.19      0.03      0.04       197
    Home Win       0.48      0.76      0.59       317

    accuracy                           0.45       760
   macro avg       0.36      0.40      0.35       760
weighted avg       0.39      0.45      0.39       760



In [22]:
def predict_match(
    stadium: str,
    competition_type: str,
    home_club_name: str,
    away_club_name: str,
    home_club_manager_name: str,
    away_club_manager_name: str,
    season: int,
):
    """Predicts match outcome and probabilities given match contextual information."""
    match_input = pd.DataFrame(
        [
            {
                "stadium": str(stadium),
                "competition_type": str(competition_type),
                "home_club_name": str(home_club_name),
                "away_club_name": str(away_club_name),
                "home_club_manager_name": str(home_club_manager_name),
                "away_club_manager_name": str(away_club_manager_name),
                "season": int(season),
            }
        ]
    )

    # Predict outcome and probabilities
    predicted_outcome = match_predictor.predict(match_input)[0]
    probabilities = match_predictor.predict_proba(match_input)[0]
    class_probs = dict(zip(match_predictor.classes_, probabilities))

    # Display clean card
    print("=" * 60)
    print(f"🏟️  FIXTURE: {home_club_name} vs {away_club_name}")
    print(f"📍  Stadium: {stadium} | Season: {season}")
    print(f"👔  Managers: {home_club_manager_name} (H) vs {away_club_manager_name} (A)")
    print("=" * 60)
    print(f"🎯  PREDICTED OUTCOME: >>> {predicted_outcome.upper()} <<<")
    print("-" * 60)
    print("📊  PROBABILITIES:")
    for outcome in ["Home Win", "Draw", "Away Win"]:
        prob_pct = class_probs.get(outcome, 0.0) * 100
        bar = "█" * int(prob_pct // 4)
        print(f"  {outcome:9s}: {prob_pct:5.1f}%  | {bar}")
    print("=" * 60 + "\n")

    return {
        "predicted_outcome": predicted_outcome,
        "probabilities": {k: f"{v*100:.1f}%" for k, v in class_probs.items()},
    }


In [23]:
predict_match(
    stadium="Etihad Stadium",
    competition_type="domestic_league",
    home_club_name="Manchester City",
    away_club_name="Arsenal FC",
    home_club_manager_name="Pep Guardiola",
    away_club_manager_name="Mikel Arteta",
    season=2025
)


🏟️  FIXTURE: Manchester City vs Arsenal FC
📍  Stadium: Etihad Stadium | Season: 2025
👔  Managers: Pep Guardiola (H) vs Mikel Arteta (A)
🎯  PREDICTED OUTCOME: >>> HOME WIN <<<
------------------------------------------------------------
📊  PROBABILITIES:
  Home Win :  70.0%  | █████████████████
  Draw     :  11.6%  | ██
  Away Win :  18.5%  | ████



{'predicted_outcome': 'Home Win',
 'probabilities': {'Away Win': '18.5%', 'Draw': '11.6%', 'Home Win': '70.0%'}}

In [24]:
predict_match(
    stadium="Anfield",
    competition_type="domestic_league",
    home_club_name="Liverpool FC",
    away_club_name="Everton FC",
    home_club_manager_name="Arne Slot",
    away_club_manager_name="Sean Dyche",
    season=2025
)


🏟️  FIXTURE: Liverpool FC vs Everton FC
📍  Stadium: Anfield | Season: 2025
👔  Managers: Arne Slot (H) vs Sean Dyche (A)
🎯  PREDICTED OUTCOME: >>> HOME WIN <<<
------------------------------------------------------------
📊  PROBABILITIES:
  Home Win :  46.9%  | ███████████
  Draw     :  38.8%  | █████████
  Away Win :  14.3%  | ███



{'predicted_outcome': 'Home Win',
 'probabilities': {'Away Win': '14.3%', 'Draw': '38.8%', 'Home Win': '46.9%'}}

In [25]:
import plotly.figure_factory as ff
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
classes = match_predictor.classes_
cm = confusion_matrix(y_test, y_pred, labels=classes)

# Format for display
z = cm.tolist()
x = [f"Pred {c}" for c in classes]
y = [f"Actual {c}" for c in classes]

fig = ff.create_annotated_heatmap(
    z, x=x, y=y, colorscale="Blues", showscale=True
)
fig.update_layout(
    title="Match Outcome Confusion Matrix (Seasons 2024-2025)",
    xaxis_title="What the Model Predicted",
    yaxis_title="What Actually Happened",
)
fig.show()


AttributeError: module 'plotly.figure_factory' has no attribute 'create_annotated_heatmap'

In [26]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Sort chronologically
compi_1 = compi_1.sort_values("date").reset_index(drop=True)
compi_1["outcome"] = np.where(
    compi_1["home_club_goals"] > compi_1["away_club_goals"],
    "Home Win",
    np.where(compi_1["home_club_goals"] < compi_1["away_club_goals"], "Away Win", "Draw"),
)

# 2. Sequential Elo Rating Engine (Zero Leakage)
INITIAL_ELO = 1500.0
K_FACTOR = 20.0
HOME_ADVANTAGE = 60.0

elo_dict = {}
home_elos, away_elos = [], []

for idx, row in compi_1.iterrows():
    h_club, a_club = row["home_club_name"], row["away_club_name"]
    h_goals, a_goals = row["home_club_goals"], row["away_club_goals"]

    # Pre-match Elo ratings
    r_h = elo_dict.get(h_club, INITIAL_ELO)
    r_a = elo_dict.get(a_club, INITIAL_ELO)
    home_elos.append(r_h)
    away_elos.append(r_a)

    # Expected outcome
    exp_h = 1.0 / (1.0 + 10.0 ** ((r_a - (r_h + HOME_ADVANTAGE)) / 400.0))
    exp_a = 1.0 - exp_h

    # Match result
    act_h = 1.0 if h_goals > a_goals else (0.5 if h_goals == a_goals else 0.0)
    act_a = 1.0 - act_h

    # Update ratings for future matches
    elo_dict[h_club] = r_h + K_FACTOR * (act_h - exp_h)
    elo_dict[a_club] = r_a + K_FACTOR * (act_a - exp_a)

compi_1["home_elo"] = home_elos
compi_1["away_elo"] = away_elos
compi_1["elo_diff"] = compi_1["home_elo"] - compi_1["away_elo"]

# 3. Rolling Form (Last 5 Games: Points & Goal Difference)
home_p = compi_1[["game_id", "date", "home_club_id", "home_club_goals", "away_club_goals", "outcome"]].copy()
home_p.columns = ["game_id", "date", "club_id", "gf", "ga", "outcome"]
home_p["pts"] = np.where(home_p["outcome"] == "Home Win", 3, np.where(home_p["outcome"] == "Draw", 1, 0))
home_p["gd"] = home_p["gf"] - home_p["ga"]
home_p["is_home"] = 1

away_p = compi_1[["game_id", "date", "away_club_id", "away_club_goals", "home_club_goals", "outcome"]].copy()
away_p.columns = ["game_id", "date", "club_id", "gf", "ga", "outcome"]
away_p["pts"] = np.where(away_p["outcome"] == "Away Win", 3, np.where(away_p["outcome"] == "Draw", 1, 0))
away_p["gd"] = away_p["gf"] - away_p["ga"]
away_p["is_home"] = 0

all_matches = pd.concat([home_p, away_p], axis=0).sort_values(["club_id", "date"]).reset_index(drop=True)
all_matches["form_pts_5"] = all_matches.groupby("club_id")["pts"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)
all_matches["form_gd_5"] = all_matches.groupby("club_id")["gd"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

h_form = all_matches[all_matches["is_home"] == 1][["game_id", "form_pts_5", "form_gd_5"]].copy()
h_form.columns = ["game_id", "home_form_pts", "home_form_gd"]

a_form = all_matches[all_matches["is_home"] == 0][["game_id", "form_pts_5", "form_gd_5"]].copy()
a_form.columns = ["game_id", "away_form_pts", "away_form_gd"]

compi_1 = compi_1.merge(h_form, on="game_id").merge(a_form, on="game_id")
compi_1["form_pts_diff"] = compi_1["home_form_pts"] - compi_1["away_form_pts"]
compi_1["form_gd_diff"] = compi_1["home_form_gd"] - compi_1["away_form_gd"]

# 4. Define Feature Sets
cat_features = [
    "stadium",
    "competition_type",
    "home_club_name",
    "away_club_name",
    "home_club_manager_name",
    "away_club_manager_name",
]
for col in cat_features:
    compi_1[col] = compi_1[col].fillna("Unknown").astype(str)

num_features = [
    "season",
    "home_elo",
    "away_elo",
    "elo_diff",
    "home_form_pts",
    "away_form_pts",
    "home_form_gd",
    "away_form_gd",
    "form_pts_diff",
    "form_gd_diff",
]

feature_cols = cat_features + num_features
X = compi_1[feature_cols]
y = compi_1["outcome"]

# 5. Temporal Train/Test Split (Seasons < 2024 train, 2024-2025 test)
train_mask = compi_1["season"] < 2024
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]

# 6. Pipeline: Impute -> Scale -> Calibrated Logistic Regression
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
        (
            "num",
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
            num_features,
        ),
    ]
)

v2_model = Pipeline(
    [
        ("prep", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, C=0.2, random_state=42)),
    ]
)

v2_model.fit(X_train, y_train)

# Evaluation
y_pred = v2_model.predict(X_test)
y_probs = v2_model.predict_proba(X_test)

print(f"✅ Upgraded Test Accuracy (Elo + Form): {accuracy_score(y_test, y_pred):.1%}")
print(f"📊 Multi-class Log Loss: {log_loss(y_test, y_probs):.3f}\n")
print(classification_report(y_test, y_pred, digits=3))


✅ Upgraded Test Accuracy (Elo + Form): 48.0%
📊 Multi-class Log Loss: 1.046

              precision    recall  f1-score   support

    Away Win      0.465     0.459     0.462       246
        Draw      0.091     0.005     0.010       197
    Home Win      0.496     0.792     0.610       317

    accuracy                          0.480       760
   macro avg      0.351     0.419     0.361       760
weighted avg      0.381     0.480     0.407       760



In [27]:
# Create lookup helpers for latest team Elo and Form
club_id_map = compi_1.set_index("home_club_name")["home_club_id"].to_dict()
latest_form_df = all_matches.groupby("club_id").last().reset_index()


def predict_match_v2(
    stadium: str,
    competition_type: str,
    home_club_name: str,
    away_club_name: str,
    home_club_manager_name: str,
    away_club_manager_name: str,
    season: int = 2025,
    bookmaker_odds: dict = None,  # e.g. {"Home Win": 2.10, "Draw": 3.40, "Away Win": 3.80}
):
    """Predicts match probabilities, fair odds, and identifies +EV value bets."""
    # 1. Fetch current dynamic Elo
    h_elo = elo_dict.get(home_club_name, 1500.0)
    a_elo = elo_dict.get(away_club_name, 1500.0)
    elo_diff = h_elo - a_elo

    # 2. Fetch latest 5-game rolling form
    h_id = club_id_map.get(home_club_name)
    a_id = club_id_map.get(away_club_name)

    h_form_pts = (
        latest_form_df[latest_form_df["club_id"] == h_id]["form_pts_5"].values[0]
        if h_id in latest_form_df["club_id"].values
        else 1.3
    )
    h_form_gd = (
        latest_form_df[latest_form_df["club_id"] == h_id]["form_gd_5"].values[0]
        if h_id in latest_form_df["club_id"].values
        else 0.0
    )

    a_form_pts = (
        latest_form_df[latest_form_df["club_id"] == a_id]["form_pts_5"].values[0]
        if a_id in latest_form_df["club_id"].values
        else 1.3
    )
    a_form_gd = (
        latest_form_df[latest_form_df["club_id"] == a_id]["form_gd_5"].values[0]
        if a_id in latest_form_df["club_id"].values
        else 0.0
    )

    input_df = pd.DataFrame(
        [
            {
                "stadium": str(stadium),
                "competition_type": str(competition_type),
                "home_club_name": str(home_club_name),
                "away_club_name": str(away_club_name),
                "home_club_manager_name": str(home_club_manager_name),
                "away_club_manager_name": str(away_club_manager_name),
                "season": int(season),
                "home_elo": h_elo,
                "away_elo": a_elo,
                "elo_diff": elo_diff,
                "home_form_pts": h_form_pts,
                "away_form_pts": a_form_pts,
                "home_form_gd": h_form_gd,
                "away_form_gd": a_form_gd,
                "form_pts_diff": h_form_pts - a_form_pts,
                "form_gd_diff": h_form_gd - a_form_gd,
            }
        ]
    )

    probs = v2_model.predict_proba(input_df)[0]
    class_probs = dict(zip(v2_model.classes_, probs))
    predicted = v2_model.predict(input_df)[0]

    # Display Value Betting Card
    print("=" * 72)
    print(f"⚽ FIXTURE: {home_club_name} vs {away_club_name}")
    print(
        f"⚡ DYNAMIC ELO: {home_club_name} ({h_elo:.0f}) vs {away_club_name} ({a_elo:.0f}) | Elo Diff: {elo_diff:+.0f}"
    )
    print(
        f"🔥 5-GAME FORM: {home_club_name} ({h_form_pts:.2f} PPG, GD {h_form_gd:+.1f}) vs {away_club_name} ({a_form_pts:.2f} PPG, GD {a_form_gd:+.1f})"
    )
    print("=" * 72)
    print(f"🎯 PREDICTED WINNER: >>> {predicted.upper()} <<<")
    print("-" * 72)
    print(
        f"{'Outcome':12s} | {'Model Prob':10s} | {'Fair Odds':10s} | {'Bookie Odds':11s} | {'Value / Edge':14s}"
    )
    print("-" * 72)

    for outcome in ["Home Win", "Draw", "Away Win"]:
        p = class_probs.get(outcome, 0.0)
        fair_odds = 1.0 / p if p > 0 else 99.0

        bookie_str = "—"
        value_str = "—"
        if bookmaker_odds and outcome in bookmaker_odds:
            b_odds = bookmaker_odds[outcome]
            ev = (p * b_odds) - 1.0
            bookie_str = f"{b_odds:.2f}"
            if ev > 0.05:
                value_str = f"+{ev*100:.1f}% (+EV ★)"
            elif ev > 0:
                value_str = f"+{ev*100:.1f}% (+EV)"
            else:
                value_str = f"{ev*100:.1f}% (No EV)"

        print(
            f"{outcome:12s} | {p*100:6.1f}%    | {fair_odds:6.2f}     | {bookie_str:11s} | {value_str:14s}"
        )
    print("=" * 72 + "\n")


In [28]:
predict_match_v2(
    stadium="Etihad Stadium",
    competition_type="domestic_league",
    home_club_name="Manchester City",
    away_club_name="Arsenal FC",
    home_club_manager_name="Pep Guardiola",
    away_club_manager_name="Mikel Arteta",
    season=2025,
    bookmaker_odds={"Home Win": 2.15, "Draw": 3.40, "Away Win": 3.75}
)


⚽ FIXTURE: Manchester City vs Arsenal FC
⚡ DYNAMIC ELO: Manchester City (1763) vs Arsenal FC (1783) | Elo Diff: -20
🔥 5-GAME FORM: Manchester City (2.20 PPG, GD +1.4) vs Arsenal FC (2.40 PPG, GD +1.0)
🎯 PREDICTED WINNER: >>> HOME WIN <<<
------------------------------------------------------------------------
Outcome      | Model Prob | Fair Odds  | Bookie Odds | Value / Edge  
------------------------------------------------------------------------
Home Win     |   63.3%    |   1.58     | 2.15        | +36.0% (+EV ★)
Draw         |   13.5%    |   7.40     | 3.40        | -54.0% (No EV)
Away Win     |   23.2%    |   4.31     | 3.75        | -12.9% (No EV)



In [29]:
predict_match_v2(
    stadium="Anfield",
    competition_type="domestic_league",
    home_club_name="Liverpool FC",
    away_club_name="Chelsea FC",
    home_club_manager_name="Arne Slot",
    away_club_manager_name="Enzo Maresca",
    season=2025
)


⚽ FIXTURE: Liverpool FC vs Chelsea FC
⚡ DYNAMIC ELO: Liverpool FC (1674) vs Chelsea FC (1599) | Elo Diff: +75
🔥 5-GAME FORM: Liverpool FC (1.40 PPG, GD +0.0) vs Chelsea FC (0.80 PPG, GD -1.0)
🎯 PREDICTED WINNER: >>> HOME WIN <<<
------------------------------------------------------------------------
Outcome      | Model Prob | Fair Odds  | Bookie Odds | Value / Edge  
------------------------------------------------------------------------
Home Win     |   54.9%    |   1.82     | —           | —             
Draw         |   21.1%    |   4.73     | —           | —             
Away Win     |   23.9%    |   4.18     | —           | —             

